# 🧠 Notebook 05: Chain-of-Thought Prompting

**Time:** 20 minutes  
**Goal:** Improve reasoning quality by making LLMs "think step-by-step"

## What is Chain-of-Thought (CoT)?

Chain-of-Thought prompting encourages LLMs to break down complex problems into intermediate steps, similar to how humans solve problems.

**Without CoT:**
Q: What is 15% of 240?
A: 36

**With CoT:**
Q: What is 15% of 240?
A: Let me work through this step by step:

Convert 15% to decimal: 15/100 = 0.15
Multiply: 0.15 × 240 = 36
Therefore, 15% of 240 is 36.


## Why Does CoT Work?

- **Breaks down complexity** - Tackles one step at a time
- **Reduces errors** - Can catch mistakes in reasoning
- **Improves accuracy** - Especially on math, logic, and multi-step problems
- **Makes reasoning transparent** - You can see where it went wrong
- **Enables verification** - Each step can be checked

## When to Use CoT

✅ **Use CoT for:**
- Math and calculations
- Multi-step reasoning
- Logic problems
- Planning and strategy
- Analysis requiring multiple steps
- Debugging and troubleshooting

❌ **Skip CoT for:**
- Simple factual questions
- Single-step tasks
- When speed matters more than accuracy
- Creative writing (unless planning plot)

Let's master it! 🚀

**Prerequisites:** Notebooks 02-04 completed

In [2]:
# Setup and Imports
import os
import sys
from pathlib import Path
import time
import json

# Add parent directory to path
notebook_dir = os.getcwd()
parent_dir = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Load environment
from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'))

# Import our modules
from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import estimate_tokens, estimate_cost, append_to_reflection
from src.config import PATH

print("=" * 60)
print("NOTEBOOK 05: CHAIN-OF-THOUGHT PROMPTING")
print("=" * 60)
print()
print(f"Configuration loaded: Path {PATH}")
print()

# Initialize client and tracker
client = LLMClient(path=PATH)
tracker = CostTracker()

print()
print("✓ Ready to learn Chain-of-Thought!")
print()

NOTEBOOK 05: CHAIN-OF-THOUGHT PROMPTING

Configuration loaded: Path A

✓ Claude API client initialized
  Default model: claude-sonnet-4-5-20250929
  Available: Opus 4.5, Sonnet 4.5, Haiku 4.5

✓ Ready to learn Chain-of-Thought!



---
## 🧪 Experiment 1: Before and After CoT

Let's see the dramatic difference CoT makes on a reasoning problem.

In [4]:
# Without vs With CoT
print("=" * 60)
print("EXPERIMENT 1: The Power of Chain-of-Thought")
print("=" * 60)
print()

problem = """A farmer has 17 sheep. All but 9 die. How many sheep are left?"""

# Without CoT
print("❌ WITHOUT Chain-of-Thought:")
print("-" * 60)

prompt_no_cot = f"{problem}\n\nAnswer:"

response_no_cot = client.generate(
    prompt=prompt_no_cot,
    temperature=0.0,
    max_tokens=50
)

if "error" not in response_no_cot:
    print(f"Prompt: {prompt_no_cot}")
    print()
    print(f"Response: {response_no_cot['content']}")
    tracker.add_call(response_no_cot)
else:
    print(f"Error: {response_no_cot['error']}")

print()
print()

# With CoT
print("✅ WITH Chain-of-Thought:")
print("-" * 60)

prompt_with_cot = f"""{problem}

Let's think through this step-by-step:"""

response_with_cot = client.generate(
    prompt=prompt_with_cot,
    temperature=0.0,
    max_tokens=150
)

if "error" not in response_with_cot:
    print(f"Prompt: {prompt_with_cot}")
    print()
    print(f"Response: {response_with_cot['content']}")
    tracker.add_call(response_with_cot)
else:
    print(f"Error: {response_with_cot['error']}")

print()
print()

print("=" * 60)
print("💡 OBSERVATION")
print("=" * 60)
print()
print("Notice how CoT:")
print("  • Shows its reasoning process")
print("  • Is more likely to get the correct answer (9 sheep)")
print("  • Makes it easy to spot errors in logic")
print("  • Builds confidence in the answer")
print()

EXPERIMENT 1: The Power of Chain-of-Thought

❌ WITHOUT Chain-of-Thought:
------------------------------------------------------------
Prompt: A farmer has 17 sheep. All but 9 die. How many sheep are left?

Answer:

Response: 9 sheep are left.

If "all but 9 die," that means 9 sheep survived.


✅ WITH Chain-of-Thought:
------------------------------------------------------------
Prompt: A farmer has 17 sheep. All but 9 die. How many sheep are left?

Let's think through this step-by-step:

Response: Let me work through this step-by-step:

**Given information:**
- The farmer starts with 17 sheep
- "All but 9 die"

**What does "all but 9" mean?**
- "All but 9" means "all except 9"
- In other words, 9 sheep did NOT die

**Answer:**
If all but 9 died, then **9 sheep are left**.

This is a bit of a trick question because our first instinct might be to subtract 9 from 17, but "all but 9 die" actually means 9 survived!


💡 OBSERVATION

Notice how CoT:
  • Shows its reasoning process
  • Is more

---
## 🎯 CoT Prompting Patterns

There are several ways to trigger Chain-of-Thought reasoning:

### Pattern 1: "Let's think step-by-step"

**Most common and effective:**
```
Problem: [your problem]

Let's think step-by-step:
```

### Pattern 2: "Let's work through this together"

**More conversational:**
```
Problem: [your problem]

Let's work through this together:
1.
```

### Pattern 3: "Before answering, show your work"

**Explicit instruction:**
```
Problem: [your problem]

Before providing your final answer, show all your work and reasoning:
```

### Pattern 4: "First, let's understand... Then..."

**Structured approach:**
```
Problem: [your problem]

First, let's understand what we're being asked.
Then, let's break it down into steps.
Finally, let's solve it.
```

In [5]:
# Testing Different CoT Patterns
print("=" * 60)
print("EXPERIMENT 2: Different CoT Patterns")
print("=" * 60)
print()

problem = """If a train travels 120 miles in 2 hours, then 180 miles in the next 3 hours, 
what is the average speed for the entire journey?"""

patterns = {
    "Pattern 1: Step-by-step": f"{problem}\n\nLet's think step-by-step:",
    
    "Pattern 2: Work together": f"{problem}\n\nLet's work through this together:",
    
    "Pattern 3: Show work": f"{problem}\n\nBefore providing your final answer, show all your work and reasoning:",
    
    "Pattern 4: Structured": f"""{problem}

First, let's understand what we're being asked.
Then, let's break it down into steps.
Finally, let's calculate the answer."""
}

for pattern_name, prompt in patterns.items():
    print(f"🎯 {pattern_name}")
    print("-" * 60)
    
    response = client.generate(
        prompt=prompt,
        temperature=0.0,
        max_tokens=200
    )
    
    if "error" not in response:
        print(response['content'][:300] + "..." if len(response['content']) > 300 else response['content'])
        tracker.add_call(response)
    else:
        print(f"Error: {response['error']}")
    
    print()
    print()
    time.sleep(0.5)

print("💡 All patterns work! Choose based on your preference and use case.")
print()

EXPERIMENT 2: Different CoT Patterns

🎯 Pattern 1: Step-by-step
------------------------------------------------------------
# Finding Average Speed for the Entire Journey

Let me work through this step-by-step.

## Step 1: Identify what we know
- First part: 120 miles in 2 hours
- Second part: 180 miles in 3 hours

## Step 2: Find total distance
Total distance = 120 miles + 180 miles = **300 miles**

## Step 3: Find tota...


🎯 Pattern 2: Work together
------------------------------------------------------------
I need to find the average speed for the entire journey.

**Average speed = Total distance ÷ Total time**

Let me identify what I know:
- First part: 120 miles in 2 hours
- Second part: 180 miles in 3 hours

**Step 1: Find the total distance**
Total distance = 120 + 180 = 300 miles

**Step 2: Find t...


🎯 Pattern 3: Show work
------------------------------------------------------------
I need to find the average speed for the entire journey.

**Given information:**
- First pa

---
## 🔢 Math and Calculations

CoT shines brightest on math problems. Let's test it!

In [6]:
# CoT for Math Problems
print("=" * 60)
print("EXPERIMENT 3: CoT for Math Problems")
print("=" * 60)
print()

math_problems = [
    "If apples cost $3 per pound and oranges cost $2 per pound, how much would 4 pounds of apples and 6 pounds of oranges cost?",
    
    "A store offers a 20% discount on items over $50, and then an additional $10 off. How much would a $80 item cost after both discounts?",
    
    "If a car uses 1 gallon of gas every 25 miles, how many gallons are needed for a 340-mile trip?"
]

for i, problem in enumerate(math_problems, 1):
    print(f"Problem {i}:")
    print("-" * 60)
    print(problem)
    print()
    
    prompt = f"{problem}\n\nLet's solve this step-by-step:"
    
    response = client.generate(
        prompt=prompt,
        temperature=0.0,
        max_tokens=200
    )
    
    if "error" not in response:
        print("Solution:")
        print(response['content'])
        tracker.add_call(response)
    else:
        print(f"Error: {response['error']}")
    
    print()
    print()
    time.sleep(0.5)

print("💡 CoT makes math problems much more reliable!")
print()

EXPERIMENT 3: CoT for Math Problems

Problem 1:
------------------------------------------------------------
If apples cost $3 per pound and oranges cost $2 per pound, how much would 4 pounds of apples and 6 pounds of oranges cost?

Solution:
I'll solve this step-by-step.

**Given information:**
- Apples cost $3 per pound
- Oranges cost $2 per pound
- Need to buy 4 pounds of apples and 6 pounds of oranges

**Step 1: Calculate the cost of apples**
- Cost of apples = 4 pounds × $3 per pound
- Cost of apples = $12

**Step 2: Calculate the cost of oranges**
- Cost of oranges = 6 pounds × $2 per pound
- Cost of oranges = $12

**Step 3: Calculate the total cost**
- Total cost = Cost of apples + Cost of oranges
- Total cost = $12 + $12
- Total cost = $24

**Answer: The total cost would be $24**


Problem 2:
------------------------------------------------------------
A store offers a 20% discount on items over $50, and then an additional $10 off. How much would a $80 item cost after both disc

---
## 🧩 Logic and Reasoning

CoT is excellent for logic puzzles and multi-step reasoning.

In [7]:
# CoT for Logic Problems
print("=" * 60)
print("EXPERIMENT 4: CoT for Logic Problems")
print("=" * 60)
print()

logic_problem = """
Three people (Alice, Bob, and Carol) are standing in a line.
- Alice is not first
- Bob is not last
- Carol is not in the middle

What is the order from first to last?
"""

print("Logic Problem:")
print(logic_problem)
print()

# Without CoT
print("WITHOUT CoT:")
print("-" * 60)

response_no_cot = client.generate(
    prompt=f"{logic_problem}\n\nAnswer:",
    temperature=0.0,
    max_tokens=50
)

if "error" not in response_no_cot:
    print(response_no_cot['content'])
    tracker.add_call(response_no_cot)

print()
print()

# With CoT
print("WITH CoT:")
print("-" * 60)

cot_prompt = f"""{logic_problem}

Let's think through this step-by-step:
1. First, let's list what we know
2. Then, let's eliminate impossible arrangements
3. Finally, let's determine the correct order"""

response_with_cot = client.generate(
    prompt=cot_prompt,
    temperature=0.0,
    max_tokens=300
)

if "error" not in response_with_cot:
    print(response_with_cot['content'])
    tracker.add_call(response_with_cot)

print()
print("💡 CoT helps break down complex constraints into manageable steps!")
print()

EXPERIMENT 4: CoT for Logic Problems

Logic Problem:

Three people (Alice, Bob, and Carol) are standing in a line.
- Alice is not first
- Bob is not last
- Carol is not in the middle

What is the order from first to last?


WITHOUT CoT:
------------------------------------------------------------
I need to work through the constraints systematically.

Let me denote the positions as: First, Middle, Last

**Constraints:**
- Alice is not first
- Bob is not last
- Carol is not in the middle


WITH CoT:
------------------------------------------------------------
# Solving the Line Order Puzzle

## Step 1: List what we know
- Alice is not first
- Bob is not last
- Carol is not in the middle

## Step 2: Eliminate impossible arrangements

Let me check all possible arrangements of three people:

1. **Alice, Bob, Carol**
   - Alice is first ❌ (violates "Alice is not first")
   
2. **Alice, Carol, Bob**
   - Alice is first ❌ (violates "Alice is not first")
   
3. **Bob, Alice, Carol**
   - Alice

---
## 🔗 Multi-Step Planning

CoT is perfect for planning tasks that require multiple sequential steps.

In [8]:
# CoT for Planning
print("=" * 60)
print("EXPERIMENT 5: CoT for Multi-Step Planning")
print("=" * 60)
print()

planning_task = """
You need to organize a small team meeting for 8 people next Tuesday at 2 PM.
You need to: book a room, send invitations, prepare an agenda, and order snacks.
The room needs AV equipment. Some team members are remote.

What should you do first, and in what order should you complete these tasks?
"""

print("Planning Task:")
print(planning_task)
print()

cot_planning_prompt = f"""{planning_task}

Let's plan this systematically:
1. First, let's identify all the dependencies
2. Then, let's determine what must be done first
3. Finally, let's create the optimal sequence"""

response = client.generate(
    prompt=cot_planning_prompt,
    temperature=0.3,
    max_tokens=400
)

if "error" not in response:
    print("CoT Planning Response:")
    print("=" * 60)
    print(response['content'])
    print("=" * 60)
    tracker.add_call(response)
else:
    print(f"Error: {response['error']}")

print()
print("💡 CoT helps identify dependencies and create logical sequences!")
print()

EXPERIMENT 5: CoT for Multi-Step Planning

Planning Task:

You need to organize a small team meeting for 8 people next Tuesday at 2 PM.
You need to: book a room, send invitations, prepare an agenda, and order snacks.
The room needs AV equipment. Some team members are remote.

What should you do first, and in what order should you complete these tasks?


CoT Planning Response:
# Meeting Planning - Task Order

## Step 1: Identify Dependencies

Let me map out what depends on what:
- **Invitations** need: room location, time (already set), agenda/purpose
- **Room booking** needs: headcount, AV requirements, date/time
- **Agenda** needs: meeting purpose (assumed known)
- **Snacks** need: headcount, dietary restrictions, room location for delivery

## Step 2: Critical Path Analysis

**Must do FIRST:**
- **Book the room** - This is the bottleneck. Without a confirmed room, you can't send complete invitations or arrange delivery.

## Step 3: Optimal Task Sequence

### **1. Book the room (DO TH

---
## 🎯 Your Turn: Practice Tasks

Time to practice using Chain-of-Thought prompting!

### 📝 Task 1: Solve a Multi-Step Problem

**Goal:** Use CoT to solve a problem that requires multiple reasoning steps.

In [10]:
# TODO - Task 1: Multi-Step Problem Solving
print("=" * 60)
print("TASK 1: Multi-Step Problem Solving")
print("=" * 60)
print()

# ============================================================================
# TODO: Create your own multi-step problem OR use one of these examples:
# ============================================================================
# YOUR PROBLEM HERE

#Example problems:
#1. "A library charges $0.50 per day for late books. Sarah returned 3 books: 
#    one was 2 days late, one was 5 days late, and one was on time. 
#   She also returned a DVD that was 3 days late at $1 per day. 
 #   What is her total late fee?"

#2. "A recipe serves 4 people and calls for 2 cups of flour, 3 eggs, and 1 cup of milk.
 #   If you want to serve 10 people and you only have 4 eggs, how much of each 
   # ingredient should you use?"

#3. "A parking garage charges $5 for the first hour, $3 for each additional hour,
    #and has a maximum daily charge of $25. If you park for 9 hours, how much do you pay?"
your_problem = """
A parking garage charges $5 for the first hour, $3 for each additional hour,
and has a maximum daily charge of $25. If you park for 9 hours, how much do you pay?
"""

# ============================================================================
# TODO: Write your CoT prompt
# ============================================================================

your_cot_prompt = f"""
{your_problem}

Let's think step-by-step. Work through the calculation one step at a time,
then state the final answer clearly on its own line as "Final answer: $X".
"""

print("Your Problem:")
print("-" * 60)
print(your_problem)
print()

print("Your CoT Prompt:")
print("-" * 60)
print(your_cot_prompt)
print()

# Test WITHOUT CoT first
print("TEST 1: Without CoT (for comparison)")
print("=" * 60)

response_without = client.generate(
    prompt=f"{your_problem}\n\nAnswer:",
    temperature=0.0,
    max_tokens=100
)

if "error" not in response_without:
    print(response_without['content'])
    tracker.add_call(response_without)
    without_answer = response_without['content']
else:
    without_answer = "Error occurred"

print()
print()

# Test WITH CoT
print("TEST 2: With CoT")
print("=" * 60)

response_with = client.generate(
    prompt=your_cot_prompt,
    temperature=0.0,
    max_tokens=300
)

if "error" not in response_with:
    print(response_with['content'])
    tracker.add_call(response_with)
    with_answer = response_with['content']
else:
    with_answer = "Error occurred"

print()
print()

# ========================================================================
# TODO: Analyze the results
# ========================================================================

print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = f"""
### Problem Chosen

[A parking garage pricing problem with a tiered rate ($5 first hour, $3 each
additional hour) and a $25 daily cap, asking the cost for 9 hours.]

### Without CoT Analysis

**Answer received:** {without_answer[:100]}...

**Correct?** [Yes]

[Even without an explicit CoT instruction, the model reasoned step by step on
its own: it computed $5 + 8×$3 = $29 and recognized that $29 exceeds the $25
maximum. However, with max_tokens=100 the response was truncated right before
it stated the final $25, so no clean final answer was given.]

### With CoT Analysis

**Reasoning shown:** 

[Step 1: cost without the cap ($5 + $24 = $29). Step 2: compare to the $25 cap
($29 > $25, so the cap applies). Step 3: conclude the charge is the cap.
]

**Final answer:** [$25]

**Correct?** [Yes]

### Comparison

**Which approach was more accurate?** [both reasoned correctly, but only the CoT version finished with a clear, correct final answer]

**Which was more transparent?** [With CoT]

**Which gave you more confidence?** [With CoT]

### Quality of Reasoning

**Did CoT show all necessary steps?** [Yes]

[What steps were shown? Any missing?It showed the base calculation, the cap comparison, and the final decision.
]

**Could you verify each step?** [Yes]

[Try to verify - were all steps correct?I checked each: $5 + $24 = $29, capped at $25 — all correct.
]

### Key Insight

[What did you learn about when CoT helps most?A strong model often reasons step-by-step even without being told to. The
bigger practical lesson here was about output length: the no-CoT version had
only 100 tokens and got cut off mid-reasoning, while the CoT version had room
to finish. Explicit CoT plus a sufficient max_tokens makes the final answer
reliable and easy to extract.]

### Real-World Application

[Where would you use this problem-solving approach?Any multi-step calculation with a constraint (billing caps, tax brackets,
shipping tiers) where I need both a correct answer and a visible, verifiable
reasoning trail.]
"""

print(reflection)

append_to_reflection(
    notebook="05",
    section_title="Task 1 - Multi-Step Problem Solving",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 1: Multi-Step Problem Solving

Your Problem:
------------------------------------------------------------

A parking garage charges $5 for the first hour, $3 for each additional hour,
and has a maximum daily charge of $25. If you park for 9 hours, how much do you pay?


Your CoT Prompt:
------------------------------------------------------------


A parking garage charges $5 for the first hour, $3 for each additional hour,
and has a maximum daily charge of $25. If you park for 9 hours, how much do you pay?


Let's think step-by-step. Work through the calculation one step at a time,
then state the final answer clearly on its own line as "Final answer: $X".


TEST 1: Without CoT (for comparison)
I need to calculate the parking cost for 9 hours.

**Breaking down the charges:**
- First hour: $5
- Additional hours: 9 - 1 = 8 hours
- Cost for additional hours: 8 × $3 = $24
- Total before maximum: $5 + $24 = $29

**Applying the maximum:**
Since $29 exceeds the maximum daily charge of $2

### 📝 Task 2: Debug Faulty Reasoning

**Goal:** Use CoT to identify where reasoning goes wrong.

**Scenario:** Sometimes LLMs make mistakes. CoT makes these visible!

In [8]:
# TODO - Task 2: Debug Faulty Reasoning
print("=" * 60)
print("TASK 2: Debug Faulty Reasoning")
print("=" * 60)
print()

# A problem designed to potentially trip up the LLM
tricky_problem = """
A bat and a ball together cost $1.10.
The bat costs $1.00 more than the ball.
How much does the ball cost?
"""

print("Tricky Problem:")
print(tricky_problem)
print()
print("(Common wrong answer: $0.10)")
print("(Correct answer: $0.05)")
print()

# ============================================================================
# TODO: Craft a CoT prompt that helps the LLM avoid the trap
# ============================================================================

debugging_prompt = f"""
# YOUR DEBUGGING PROMPT HERE

Example:
{tricky_problem}

Let's think very carefully step-by-step:
1. First, let's define our variables
2. Then, let's write out the equations
3. Then, let's solve systematically
4. Finally, let's verify our answer
"""

response = client.generate(
    prompt=debugging_prompt,
    temperature=0.0,
    max_tokens=300
)

if "error" not in response:
    print("CoT Response:")
    print("=" * 60)
    print(response['content'])
    print("=" * 60)
    tracker.add_call(response)
    
    # ========================================================================
    # TODO: Analyze if it got the right answer
    # ========================================================================
    
    print()
    print("=" * 60)
    print("REFLECTION")
    print("=" * 60)
    print()
    
    reflection = """
### Did it get the correct answer? ($0.05)

[Yes/No]

### Reasoning Quality

**Were the steps correct?**

[Analyze each step shown]

Step 1: [Correct/Incorrect - explain]
Step 2: [Correct/Incorrect - explain]
Step 3: [Correct/Incorrect - explain]
...

### If it got it wrong:

**Where did the reasoning fail?**

[Identify the specific step where it went wrong]

**How would you fix the prompt to help it?**

[Your improved prompt strategy]

### If it got it right:

**What made the CoT effective?**

[What about your prompt helped avoid the trap?]

**Could you simplify the prompt and still get it right?**

[Try it - test a simpler version]

### Key Learning

**What makes a problem "tricky"?**

[Your analysis]

**How does CoT help with tricky problems?**

[Your insights]

### Debugging Strategy

**How would you use CoT to debug LLM reasoning in production?**

[Describe your approach]
"""
    
    print(reflection)
    
    append_to_reflection(
        notebook="05",
        section_title="Task 2 - Debug Faulty Reasoning",
        reflection_content=reflection,
        output_dir=os.path.join(parent_dir, 'outputs')
    )
    
    print()
    print("💾 Reflection saved to outputs/homework_reflection.md")

else:
    print(f"Error: {response['error']}")

print()

TASK 2: Debug Faulty Reasoning

Tricky Problem:

A bat and a ball together cost $1.10.
The bat costs $1.00 more than the ball.
How much does the ball cost?


(Common wrong answer: $0.10)
(Correct answer: $0.05)

CoT Response:
I'll solve this classic problem step-by-step.

## Step 1: Define our variables
Let me define:
- Let b = cost of the ball (in dollars)
- Let B = cost of the bat (in dollars)

## Step 2: Write out the equations
From the problem, I can write two equations:
- Equation 1: B + b = 1.10 (together they cost $1.10)
- Equation 2: B = b + 1.00 (the bat costs $1.00 more than the ball)

## Step 3: Solve systematically
Now I'll substitute Equation 2 into Equation 1:
- (b + 1.00) + b = 1.10
- 2b + 1.00 = 1.10
- 2b = 1.10 - 1.00
- 2b = 0.10
- b = 0.05

So the ball costs $0.05 (5 cents)

Now I can find the bat's cost:
- B = b + 1.00
- B = 0.05 + 1.00
- B = 1.05

So the bat costs $1.05

## Step 4: Verify our answer
Let me

REFLECTION


### Did it get the correct answer? ($0.05)

[Y

### 📝 Task 3: Compare CoT Across Different Models/Temperatures

**Goal:** Understand how CoT interacts with model settings.

**Experiment:** Test how temperature affects CoT quality.

In [9]:
# TODO - Task 3: CoT with Different Settings
print("=" * 60)
print("TASK 3: CoT Across Different Settings")
print("=" * 60)
print()

test_problem = """
A store has a sale: "Buy 2, get 1 free" on items that cost $15 each.
If you want to get 7 items, how much will you pay?
"""

print("Problem:")
print(test_problem)
print()

cot_prompt_base = f"""{test_problem}

Let's work through this step-by-step:
"""

# Test different temperatures
temperatures = [0.0, 0.5, 1.0]

results = {}

for temp in temperatures:
    print(f"Testing with temperature={temp}")
    print("-" * 60)
    
    response = client.generate(
        prompt=cot_prompt_base,
        temperature=temp,
        max_tokens=250
    )
    
    if "error" not in response:
        print(response['content'][:200] + "..." if len(response['content']) > 200 else response['content'])
        results[temp] = response['content']
        tracker.add_call(response)
    else:
        print(f"Error: {response['error']}")
        results[temp] = "Error"
    
    print()
    print()
    time.sleep(0.5)

# ========================================================================
# TODO: Analyze the differences
# ========================================================================

print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = f"""
### Temperature Comparison

#### Temperature 0.0 (Deterministic)

**Reasoning quality:** [Rate 1-5]: ___/5

**Final answer:** [What answer did it give?]

**Consistency:** [If you ran it again, would it be the same?]

**Observations:**
[What did you notice about the reasoning?]

#### Temperature 0.5 (Balanced)

**Reasoning quality:** [Rate 1-5]: ___/5

**Final answer:** [What answer did it give?]

**Different from temp 0.0?** [Yes/No - how?]

**Observations:**
[What changed with more randomness?]

#### Temperature 1.0 (Creative)

**Reasoning quality:** [Rate 1-5]: ___/5

**Final answer:** [What answer did it give?]

**Different from others?** [Yes/No - how?]

**Observations:**
[Was the reasoning still logical?]

### Overall Analysis

**Best temperature for CoT reasoning:** [0.0 / 0.5 / 1.0]

**Why?**
[Your reasoning]

**Does CoT work better with lower temperatures?**
[Yes/No - explain your findings]

### Trade-offs

**Accuracy vs Creativity:**
[How did temperature affect the balance?]

**When might you want higher temperature with CoT?**
[Describe a scenario]

**When should you always use temperature 0.0?**
[Describe scenarios]

### Production Recommendations

**For math/logic problems:** Temperature = ___

**For planning/strategy:** Temperature = ___

**For creative problem-solving:** Temperature = ___

**Reasoning:** [Explain your recommendations]
"""

print(reflection)

append_to_reflection(
    notebook="05",
    section_title="Task 3 - CoT with Different Settings",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 3: CoT Across Different Settings

Problem:

A store has a sale: "Buy 2, get 1 free" on items that cost $15 each.
If you want to get 7 items, how much will you pay?


Testing with temperature=0.0
------------------------------------------------------------
I need to figure out how much you'll pay for 7 items under a "Buy 2, get 1 free" promotion where each item costs $15.

**Understanding the promotion:**
- For every 2 items you buy, you get 1 free
- So...


Testing with temperature=0.5
------------------------------------------------------------
I need to figure out how much you'll pay for 7 items under a "Buy 2, get 1 free" promotion where each item costs $15.

**Understanding the promotion:**
- For every 2 items you buy, you get 1 free
- So...


Testing with temperature=1.0
------------------------------------------------------------
I need to figure out how many items I pay for when getting 7 items with a "Buy 2, get 1 free" promotion.

**Understanding the promotion:**
- For ev

---
## 🎓 Advanced CoT Techniques

In [10]:
# Advanced CoT Patterns
print("=" * 60)
print("ADVANCED COT TECHNIQUES")
print("=" * 60)
print()

advanced_techniques = {
    "Self-Consistency": """
    Run the same CoT prompt multiple times (with temp > 0) and take the majority answer.
    
    Example:
    - Run 5 times
    - Get answers: [42, 42, 43, 42, 42]
    - Final answer: 42 (appears 4/5 times)
    
    Use when: High stakes decisions, uncertain problems
    """,
    
    "Least-to-Most": """
    Break complex problems into simpler sub-problems, solve them in order.
    
    Example:
    "Let's start with the simplest part:
     1. First, solve for X
     2. Using X, now solve for Y
     3. Finally, combine X and Y to get Z"
    
    Use when: Hierarchical problems, building up complexity
    """,
    
    "Tree of Thoughts": """
    Explore multiple reasoning paths, then choose the best.
    
    Example:
    "Let's explore two approaches:
     Approach 1: [reasoning path 1]
     Approach 2: [reasoning path 2]
     
     Now let's evaluate which is better..."
    
    Use when: Multiple valid approaches, complex trade-offs
    """,
    
    "Verification": """
    Add explicit verification steps.
    
    Example:
    "Let's solve step-by-step:
     [steps]
     
     Now let's verify our answer:
     [check each step]
     [plug back into original problem]"
    
    Use when: Critical applications, error-prone calculations
    """,
    
    "Reflection": """
    Ask the model to reflect on its own reasoning.
    
    Example:
    "After showing your reasoning:
     1. Identify any assumptions you made
     2. Consider if there are errors
     3. Suggest improvements to your approach"
    
    Use when: Learning, improvement, critical analysis
    """
}

for technique, description in advanced_techniques.items():
    print(f"🎯 {technique}")
    print("-" * 60)
    print(description)
    print()

print()
print("💡 These advanced techniques can significantly improve reasoning quality!")
print("   Experiment with them in your projects.")
print()

ADVANCED COT TECHNIQUES

🎯 Self-Consistency
------------------------------------------------------------

    Run the same CoT prompt multiple times (with temp > 0) and take the majority answer.

    Example:
    - Run 5 times
    - Get answers: [42, 42, 43, 42, 42]
    - Final answer: 42 (appears 4/5 times)

    Use when: High stakes decisions, uncertain problems
    

🎯 Least-to-Most
------------------------------------------------------------

    Break complex problems into simpler sub-problems, solve them in order.

    Example:
    "Let's start with the simplest part:
     1. First, solve for X
     2. Using X, now solve for Y
     3. Finally, combine X and Y to get Z"

    Use when: Hierarchical problems, building up complexity
    

🎯 Tree of Thoughts
------------------------------------------------------------

    Explore multiple reasoning paths, then choose the best.

    Example:
    "Let's explore two approaches:
     Approach 1: [reasoning path 1]
     Approach 2: [reaso

---
## 📊 CoT Best Practices

In [11]:
# CoT Best Practices
print("=" * 60)
print("CHAIN-OF-THOUGHT BEST PRACTICES")
print("=" * 60)
print()

best_practices = {
    "When to Use CoT": [
        "✓ Multi-step problems (math, logic, planning)",
        "✓ When accuracy is critical",
        "✓ When you need to verify reasoning",
        "✓ When the problem is complex or tricky",
        "✓ When debugging incorrect answers",
        "✗ Simple factual questions",
        "✗ Single-step tasks",
        "✗ When speed is more important than accuracy"
    ],
    
    "Prompting Techniques": [
        "✓ Use explicit phrases: 'Let's think step-by-step'",
        "✓ Guide the structure: numbered steps, categories",
        "✓ Ask for verification: 'Now check your work'",
        "✓ Request final answer separately: 'Therefore, the answer is...'",
        "✓ Use temperature 0.0 for consistency",
        "✗ Don't assume CoT will happen automatically",
        "✗ Don't leave reasoning structure vague"
    ],
    
    "Parsing CoT Outputs": [
        "✓ Look for clear step markers (1., 2., 3.)",
        "✓ Extract final answer explicitly",
        "✓ Verify each step if critical",
        "✓ Store reasoning for debugging",
        "✓ Use structured formats when possible",
        "✗ Don't just grab the last number",
        "✗ Don't skip verification on critical tasks"
    ],
    
    "Production Use": [
        "✓ Cache successful CoT patterns",
        "✓ Monitor reasoning quality over time",
        "✓ A/B test with and without CoT",
        "✓ Combine with structured output for final answer",
        "✓ Implement verification for high-stakes decisions",
        "✗ Don't use CoT everywhere (cost/latency)",
        "✗ Don't trust reasoning blindly"
    ]
}

for category, practices in best_practices.items():
    print(f"📌 {category}")
    print("-" * 60)
    for practice in practices:
        print(f"  {practice}")
    print()

print()
print("=" * 60)
print("GOLDEN RULES FOR COT")
print("=" * 60)
print()
print("1. CoT works best with temperature 0.0")
print("2. Always explicitly request step-by-step reasoning")
print("3. Verify critical outputs - don't trust blindly")
print("4. Use CoT for accuracy, skip for speed")
print("5. Combine CoT with structured output for best results")
print()

CHAIN-OF-THOUGHT BEST PRACTICES

📌 When to Use CoT
------------------------------------------------------------
  ✓ Multi-step problems (math, logic, planning)
  ✓ When accuracy is critical
  ✓ When you need to verify reasoning
  ✓ When the problem is complex or tricky
  ✓ When debugging incorrect answers
  ✗ Simple factual questions
  ✗ Single-step tasks
  ✗ When speed is more important than accuracy

📌 Prompting Techniques
------------------------------------------------------------
  ✓ Use explicit phrases: 'Let's think step-by-step'
  ✓ Guide the structure: numbered steps, categories
  ✓ Ask for verification: 'Now check your work'
  ✓ Request final answer separately: 'Therefore, the answer is...'
  ✓ Use temperature 0.0 for consistency
  ✗ Don't assume CoT will happen automatically
  ✗ Don't leave reasoning structure vague

📌 Parsing CoT Outputs
------------------------------------------------------------
  ✓ Look for clear step markers (1., 2., 3.)
  ✓ Extract final answer explici

---
## ✅ Notebook 05 Complete!

### Summary

You've mastered Chain-of-Thought prompting! You now know:
- ✅ What CoT is and why it works
- ✅ Multiple CoT prompting patterns
- ✅ When to use (and not use) CoT
- ✅ How to debug reasoning with CoT
- ✅ Advanced CoT techniques
- ✅ Production best practices

In [12]:
# Final Reflection
print("=" * 60)
print("OVERALL NOTEBOOK REFLECTION")
print("=" * 60)
print()

# ============================================================================
# TODO: Final reflection on Chain-of-Thought
# ============================================================================

reflection = """
### 1. How has CoT changed your approach to complex problems?

[Your answer]

### 2. When will you use CoT vs regular prompting?

[Your decision criteria]

### 3. What's your confidence in using CoT? (1-5)

**Confidence:** ___/5

[What would increase it?]

### 4. Most surprising finding about CoT?

[Your biggest "aha!" moment]

### 5. How will you use CoT in your projects?

[Real-world applications]

### 6. CoT limitations you discovered?

[What doesn't work well with CoT?]

### 7. Key takeaway from this notebook?

[Your main learning]
"""

print(reflection)

# Save reflection
append_to_reflection(
    notebook="05",
    section_title="Overall Reflection",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")

# Show costs
print()
print("=" * 60)
print("YOUR COSTS THIS NOTEBOOK")
print("=" * 60)
print()
tracker.report()

print()
print("=" * 60)
print("✅ NOTEBOOK 05 COMPLETE!")
print("=" * 60)
print()
print("Progress: [████████████████░░░░] 63% Complete")
print()
print("✓ Notebook 00: Setup Verification")
print("✓ Notebook 01: Environment Setup")
print("✓ Notebook 02: LLM Basics")
print("✓ Notebook 03: CO-STAR Framework")
print("✓ Notebook 04: Structured Outputs")
print("✓ Notebook 05: Chain of Thought ← YOU ARE HERE")
print("○ Notebook 06: Model Comparison")
print("○ Notebook 07: MCP Introduction")
print("○ Notebook 08: Project Kickoff")
print()
print("Next: notebooks/06_model_comparison.ipynb")
print()

OVERALL NOTEBOOK REFLECTION


### 1. How has CoT changed your approach to complex problems?

[Your answer]

### 2. When will you use CoT vs regular prompting?

[Your decision criteria]

### 3. What's your confidence in using CoT? (1-5)

**Confidence:** ___/5

[What would increase it?]

### 4. Most surprising finding about CoT?

[Your biggest "aha!" moment]

### 5. How will you use CoT in your projects?

[Real-world applications]

### 6. CoT limitations you discovered?

[What doesn't work well with CoT?]

### 7. Key takeaway from this notebook?

[Your main learning]


💾 Reflection saved to outputs/homework_reflection.md

YOUR COSTS THIS NOTEBOOK

💰 API COST REPORT
Total API calls: 18
Total input tokens: 1,330
Total output tokens: 3,566
Total cost: $0.0575

Recent calls:
  1. [22:46:55] sonnet - 48in/300out - $0.0046
  2. [22:47:56] sonnet - 107in/300out - $0.0048
  3. [22:48:35] sonnet - 64in/229out - $0.0036
  4. [22:48:40] sonnet - 64in/250out - $0.0039
  5. [22:48:45] sonnet - 64in/2